# 09｜Choice财务报表与分红小样本验收（官方CTR修复版）

本Notebook依据Choice官方量化API技能包修复，**不再使用`css + 候选指标试探`**：

- 利润表：`c.ctr("IncomeStatementSHSZ", ...)`；
- 资产负债表：`c.ctr("BalanceStatementSHSZ", ...)`；
- 现金流量表：`c.ctr("CashFlowStatementSHSZ", ...)`；
- 分红实施：`c.ctr("DividendImplementationInfo", ...)`。

默认只抓取3只证券和2个报告期，不进行全市场下载：

- `000001.SZ`、`600519.SH`、`300750.SZ`；
- `2025-12-31`、`2026-06-30`；
- 财务报表使用`ReportType=1`（合并报表）；
- 分红按`DateType=3`（报告期）筛选。

重要变化：财务指标单位按官方报表明确保存为`CNY`；分红字段按官方说明分别保存为`CNY/share`、`10k_share`、`date`、`text`或`vendor_raw_ratio`。空值不会填0。

本版只调用一次Choice API。第二次幂等验收只重复执行本地SQLite upsert，不重复消耗接口额度。


In [12]:
from pathlib import Path
import os
import sys


def locate_project_root():
    configured = os.getenv("QIANJI_PROJECT_ROOT", "").strip()
    candidates = [Path(configured)] if configured else []
    current = Path.cwd().resolve()
    candidates.extend([current, current.parent, *current.parents])
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "qianji_data_mini").exists():
            return candidate.resolve()
    raise RuntimeError("未找到项目根目录。请设置QIANJI_PROJECT_ROOT。")


PROJECT_ROOT = locate_project_root()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env", override=True)

print("Python路径：", sys.executable)
print("项目根目录：", PROJECT_ROOT)
print("Notebook当前目录：", Path.cwd().resolve())


Python路径： d:\minicoda3\envs\dm311\python.exe
项目根目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini
Notebook当前目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\notebooks


In [13]:
from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version

try:
    installed_version = version("qianji-data-mini")
except PackageNotFoundError:
    installed_version = "0.0.0"

print("qianji-data-mini版本：", installed_version)
if Version(installed_version) < Version("0.8.0"):
    raise RuntimeError("当前版本低于0.8.0。请运行00号Notebook重新安装项目后再运行。")

try:
    from EmQuantAPI import c
    print("EmQuantAPI：导入成功")
except Exception as exc:
    raise RuntimeError(f"EmQuantAPI导入失败：{type(exc).__name__}: {exc}") from exc


qianji-data-mini版本： 0.8.0
EmQuantAPI：导入成功


## 官方报表与字段映射

这些报表名、字段名和参数来自Choice官方技能包。财务字段选取一般企业和商业银行均存在的核心指标，因此3只样本可以使用同一套字段验收。


In [14]:
from datetime import date, datetime, timezone


def csv_items(name, defaults):
    raw = os.getenv(name, ",".join(defaults))
    return list(dict.fromkeys(item.strip().upper() for item in raw.split(",") if item.strip()))


SYMBOLS = csv_items("CHOICE_FINANCIAL_SAMPLE_SYMBOLS", ["000001.SZ", "600519.SH", "300750.SZ"])
REPORT_DATES = csv_items("CHOICE_FINANCIAL_REPORT_DATES", ["2025-12-31", "2026-06-30"])

REPORT_SPECS = {
    "income": {
        "ctr_name": "IncomeStatementSHSZ",
        "indicators": csv_items(
            "CHOICE_CTR_INCOME_FIELDS",
            ["OPERATEREVE", "NETPROFIT", "PARENTNETPROFIT"],
        ),
    },
    "balance": {
        "ctr_name": "BalanceStatementSHSZ",
        "indicators": csv_items(
            "CHOICE_CTR_BALANCE_FIELDS",
            ["SUMASSET", "SUMLIAB", "SUMSHEQUITY"],
        ),
    },
    "cashflow": {
        "ctr_name": "CashFlowStatementSHSZ",
        "indicators": csv_items(
            "CHOICE_CTR_CASHFLOW_FIELDS",
            ["NETOPERATECASHFLOW", "NETINVCASHFLOW", "NETFINACASHFLOW", "NICASHEQUI"],
        ),
    },
}

DIVIDEND_SPEC = {
    "ctr_name": "DividendImplementationInfo",
    "indicators": csv_items(
        "CHOICE_CTR_DIVIDEND_FIELDS",
        [
            "DIVWAY", "DIVCASHPSBFTAX", "DIVCASHPSAFTAX",
            "DIVSTOCKPSRATIO", "DIVCAPITPSRATIO", "DIVRTISSBASESHARES",
            "SHAREBASEDATE", "DIVIMPLANNCDATE", "DIVRECORDDATE",
            "DIVEXDATE", "DIVPAYDATE",
        ],
    ),
}

DIVIDEND_UNITS = {
    "DIVWAY": "text",
    "DIVCASHPSBFTAX": "CNY/share",
    "DIVCASHPSAFTAX": "CNY/share",
    "DIVSTOCKPSRATIO": "vendor_raw_ratio",
    "DIVCAPITPSRATIO": "vendor_raw_ratio",
    "DIVRTISSBASESHARES": "10k_share",
    "SHAREBASEDATE": "date",
    "DIVIMPLANNCDATE": "date",
    "DIVRECORDDATE": "date",
    "DIVEXDATE": "date",
    "DIVPAYDATE": "date",
}

REPORT_TYPE = int(os.getenv("CHOICE_CTR_REPORT_TYPE", "1"))
RUN_IDEMPOTENCY_CHECK = os.getenv("CHOICE_FINANCIAL_RUN_TWICE", "1") == "1"
CREATE_BACKUP = os.getenv("CHOICE_FINANCIAL_CREATE_BACKUP", "1") == "1"
STRICT_MODE = os.getenv("CHOICE_FINANCIAL_STRICT", "0") == "1"
ALLOW_LARGE_SAMPLE = os.getenv("CHOICE_FINANCIAL_ALLOW_LARGE_SAMPLE", "0") == "1"

configured_db = Path(os.getenv("QIANJI_DB_PATH", "data/qianji_market.db"))
DATABASE_PATH = configured_db.resolve() if configured_db.is_absolute() else (PROJECT_ROOT / configured_db).resolve()
OUTPUT_DIR = PROJECT_ROOT / "validation_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if REPORT_TYPE not in {1, 2, 3, 4}:
    raise ValueError("CHOICE_CTR_REPORT_TYPE只能是1、2、3或4；日常验收建议使用1（合并报表）。")
if not ALLOW_LARGE_SAMPLE and (len(SYMBOLS) > 5 or len(REPORT_DATES) > 8):
    raise ValueError("09号默认只允许最多5只证券、8个报告期；不要直接改成全市场下载。")
for value in REPORT_DATES:
    date.fromisoformat(value)

safe_config = {
    "notebook_version": "0.7.1-ctr",
    "symbols": SYMBOLS,
    "report_dates": REPORT_DATES,
    "report_type": REPORT_TYPE,
    "report_specs": REPORT_SPECS,
    "dividend_spec": DIVIDEND_SPEC,
    "idempotency_mode": "local_upsert_only",
    "database_path": str(DATABASE_PATH),
    "choice_login_mode": os.getenv("CHOICE_LOGIN_MODE", "auto"),
    "choice_username_configured": bool(os.getenv("CHOICE_USERNAME")),
    "choice_password_configured": bool(os.getenv("CHOICE_PASSWORD")),
}
safe_config


{'notebook_version': '0.7.1-ctr',
 'symbols': ['000001.SZ', '600519.SH', '300750.SZ'],
 'report_dates': ['2025-12-31', '2026-06-30'],
 'report_type': 1,
 'report_specs': {'income': {'ctr_name': 'IncomeStatementSHSZ',
   'indicators': ['OPERATEREVE', 'NETPROFIT', 'PARENTNETPROFIT']},
  'balance': {'ctr_name': 'BalanceStatementSHSZ',
   'indicators': ['SUMASSET', 'SUMLIAB', 'SUMSHEQUITY']},
  'cashflow': {'ctr_name': 'CashFlowStatementSHSZ',
   'indicators': ['NETOPERATECASHFLOW',
    'NETINVCASHFLOW',
    'NETFINACASHFLOW',
    'NICASHEQUI']}},
 'dividend_spec': {'ctr_name': 'DividendImplementationInfo',
  'indicators': ['DIVWAY',
   'DIVCASHPSBFTAX',
   'DIVCASHPSAFTAX',
   'DIVSTOCKPSRATIO',
   'DIVCAPITPSRATIO',
   'DIVRTISSBASESHARES',
   'SHAREBASEDATE',
   'DIVIMPLANNCDATE',
   'DIVRECORDDATE',
   'DIVEXDATE',
   'DIVPAYDATE']},
 'idempotency_mode': 'local_upsert_only',
 'database_path': 'D:\\OneDrive\\桌面\\数据基座代码\\qianji_openbb_mini\\data\\qianji_market.db',
 'choice_login_mode': 

## 数据库备份与运行前计数


In [15]:
import sqlite3

from qianji_data_mini import Database

database = Database(DATABASE_PATH)
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = None
if CREATE_BACKUP and DATABASE_PATH.exists():
    backup_dir = OUTPUT_DIR / "backups"
    backup_dir.mkdir(parents=True, exist_ok=True)
    backup_path = backup_dir / f"qianji_market_before_choice_financial_{run_timestamp}.db"
    with database.connect() as source_connection, sqlite3.connect(backup_path) as target_connection:
        source_connection.backup(target_connection)
    print("数据库一致性备份：", backup_path)


def scalar(sql, params=()):
    with database.connect() as connection:
        return connection.execute(sql, params).fetchone()[0]


def scoped_counts(stage):
    symbol_placeholders = ",".join("?" for _ in SYMBOLS)
    date_placeholders = ",".join("?" for _ in REPORT_DATES)
    params = [*SYMBOLS, *REPORT_DATES]
    return {
        "stage": stage,
        "statement_scope": scalar(
            f"SELECT COUNT(*) FROM financial_statement_fact WHERE source='choice' "
            f"AND symbol IN ({symbol_placeholders}) AND report_date IN ({date_placeholders})",
            params,
        ),
        "dividend_scope": scalar(
            f"SELECT COUNT(*) FROM dividend_fact WHERE source='choice' "
            f"AND symbol IN ({symbol_placeholders}) AND report_date IN ({date_placeholders})",
            params,
        ),
        "financial_run_log": scalar("SELECT COUNT(*) FROM financial_ingestion_run"),
    }


counts_before = scoped_counts("刷新前")
counts_before


数据库一致性备份： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\backups\qianji_market_before_choice_financial_20260902_132440.db


{'stage': '刷新前',
 'statement_scope': 60,
 'dividend_scope': 44,
 'financial_run_log': 3}

## 官方CTR查询、解析与落库函数

本单元只定义函数，不会登录或请求Choice。`Ispandas=1`始终放在`options`末尾，避免CTR返回类型判断失效。


In [16]:
import json
import math
import pandas as pd

from qianji_data_mini.adapters.choice import ChoiceAdapter, normalize_choice_date
from qianji_data_mini.models import DividendFact, FinancialStatementFact


EMPTY_MARKERS = {"", "--", "NONE", "NULL", "NAN"}


def split_fact_value(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None, None
    text = str(value).strip()
    if text.upper() in EMPTY_MARKERS:
        return None, None
    try:
        numeric = float(text.replace(",", ""))
        if not math.isfinite(numeric):
            numeric = None
    except (TypeError, ValueError):
        numeric = None
    return numeric, text


def normalize_optional_date(value):
    if value is None or str(value).strip().upper() in EMPTY_MARKERS:
        return None
    try:
        return normalize_choice_date(value)
    except Exception:
        return None


def ctr_dataframe(adapter, ctr_name, indicators, options, request_id, audit_rows):
    response = adapter.c.ctr(ctr_name, ",".join(indicators), options)
    if not isinstance(response, pd.DataFrame):
        code_value = getattr(response, "ErrorCode", -1)
        message = getattr(response, "ErrorMsg", "未返回错误说明")
        audit_rows.append({
            "request": request_id, "ctr_name": ctr_name, "status": "error",
            "rows": 0, "error_code": code_value, "error": message, "options": options,
        })
        raise RuntimeError(f"{ctr_name}失败（ErrorCode={code_value}）：{message}")

    frame = response.copy()
    frame.columns = [str(column).strip().upper() for column in frame.columns]
    audit_rows.append({
        "request": request_id, "ctr_name": ctr_name,
        "status": "success" if not frame.empty else "success_empty",
        "rows": len(frame), "error_code": 0, "error": "", "options": options,
    })
    return frame


def fetch_official_ctr_sample():
    statement_records = []
    dividend_records = []
    audit_rows = []
    errors = {}
    missing_fields = {"income": {}, "balance": {}, "cashflow": {}, "dividend": {}}
    adapter = ChoiceAdapter()
    try:
        for symbol in SYMBOLS:
            for report_date_text in REPORT_DATES:
                requested_date = date.fromisoformat(report_date_text)
                for statement_type, spec in REPORT_SPECS.items():
                    request_id = f"{statement_type}:{symbol}:{report_date_text}"
                    indicators = ["REPORTDATE", *spec["indicators"]]
                    options = (
                        f"SecuCode={symbol},ReportDate={report_date_text},"
                        f"ReportType={REPORT_TYPE},RECVtimeout=60,Ispandas=1"
                    )
                    try:
                        frame = ctr_dataframe(
                            adapter, spec["ctr_name"], indicators, options, request_id, audit_rows
                        )
                        if frame.empty:
                            raise RuntimeError("请求成功但报表为空")
                        if "REPORTDATE" in frame.columns:
                            normalized_dates = frame["REPORTDATE"].map(normalize_optional_date)
                            matched = frame.loc[normalized_dates == requested_date]
                            row = (matched if not matched.empty else frame).iloc[-1]
                        else:
                            row = frame.iloc[-1]

                        missing = [field for field in spec["indicators"] if field not in frame.columns]
                        for field in missing:
                            missing_fields[statement_type][field] = "官方字段未出现在当前账号返回列中"

                        raw_row = {str(key): value for key, value in row.to_dict().items()}
                        for indicator in spec["indicators"]:
                            if indicator not in frame.columns:
                                continue
                            numeric, text = split_fact_value(row[indicator])
                            statement_records.append(FinancialStatementFact(
                                symbol=symbol,
                                statement_type=statement_type,
                                report_date=requested_date,
                                indicator=indicator,
                                value_numeric=numeric,
                                value_text=text,
                                currency="CNY",
                                unit="CNY",
                                raw={"ctr_name": spec["ctr_name"], "row": raw_row},
                            ))
                    except Exception as exc:
                        errors[request_id] = f"{type(exc).__name__}: {exc}"

            dividend_request = f"dividend:{symbol}:{min(REPORT_DATES)}:{max(REPORT_DATES)}"
            dividend_fields = ["SECUCODE", "REPORTDATE", *DIVIDEND_SPEC["indicators"]]
            options = (
                f"secucode={symbol},StartDate={min(REPORT_DATES)},EndDate={max(REPORT_DATES)},"
                "DateType=3,RECVtimeout=60,Ispandas=1"
            )
            try:
                frame = ctr_dataframe(
                    adapter, DIVIDEND_SPEC["ctr_name"], dividend_fields,
                    options, dividend_request, audit_rows,
                )
                if frame.empty:
                    continue
                if "REPORTDATE" not in frame.columns:
                    raise RuntimeError("分红报表缺少REPORTDATE列")

                frame["_REPORTDATE"] = frame["REPORTDATE"].map(normalize_optional_date)
                frame = frame[frame["_REPORTDATE"].isin({date.fromisoformat(item) for item in REPORT_DATES})].copy()
                if "DIVIMPLANNCDATE" in frame.columns:
                    frame["_SORT_DATE"] = frame["DIVIMPLANNCDATE"].map(normalize_optional_date)
                    frame = frame.sort_values("_SORT_DATE", na_position="first")
                frame = frame.drop_duplicates("_REPORTDATE", keep="last")

                missing = [field for field in DIVIDEND_SPEC["indicators"] if field not in frame.columns]
                for field in missing:
                    missing_fields["dividend"][field] = "官方字段未出现在当前账号返回列中"

                for _, row in frame.iterrows():
                    report_date_value = row["_REPORTDATE"]
                    if report_date_value is None:
                        continue
                    raw_row = {
                        str(key): value for key, value in row.drop(labels=["_REPORTDATE", "_SORT_DATE"], errors="ignore").to_dict().items()
                    }
                    for indicator in DIVIDEND_SPEC["indicators"]:
                        if indicator not in frame.columns:
                            continue
                        numeric, text = split_fact_value(row[indicator])
                        dividend_records.append(DividendFact(
                            symbol=symbol,
                            report_date=report_date_value,
                            indicator=indicator,
                            value_numeric=numeric,
                            value_text=text,
                            currency="CNY",
                            unit=DIVIDEND_UNITS.get(indicator, "vendor_raw"),
                            raw={"ctr_name": DIVIDEND_SPEC["ctr_name"], "row": raw_row},
                        ))
            except Exception as exc:
                errors[dividend_request] = f"{type(exc).__name__}: {exc}"
    finally:
        adapter.close()

    return statement_records, dividend_records, pd.DataFrame(audit_rows), missing_fields, errors


## 一次真实下载与两次本地幂等落库

本单元会登录Choice并真实请求。不会打印账号、密码、Token或userinfo。第二次只重复写入同一批内存记录，不再次请求Choice。


In [17]:
started_at = datetime.now(timezone.utc)
statement_records, dividend_records, request_audit, rejected_indicators, request_errors = fetch_official_ctr_sample()

selected_indicators = {
    statement_type: sorted({item.indicator for item in statement_records if item.statement_type == statement_type})
    for statement_type in REPORT_SPECS
}
selected_indicators["dividend"] = sorted({item.indicator for item in dividend_records})

first_statement_stored = database.upsert_financial_statement_facts(statement_records)
first_dividend_stored = database.upsert_dividend_facts(dividend_records)
counts_after_first = scoped_counts("第一次落库后")

if RUN_IDEMPOTENCY_CHECK:
    second_statement_stored = database.upsert_financial_statement_facts(statement_records)
    second_dividend_stored = database.upsert_dividend_facts(dividend_records)
    counts_after_second = scoped_counts("第二次本地upsert后")
else:
    second_statement_stored = 0
    second_dividend_stored = 0
    counts_after_second = scoped_counts("未执行第二次本地upsert")

finished_at = datetime.now(timezone.utc)
database.log_financial_ingestion(
    source="choice",
    requested_symbols=SYMBOLS,
    requested_report_dates=REPORT_DATES,
    selected_indicators=selected_indicators,
    rejected_indicators=rejected_indicators,
    statement_received_rows=len(statement_records),
    statement_stored_rows=first_statement_stored,
    dividend_received_rows=len(dividend_records),
    dividend_stored_rows=first_dividend_stored,
    errors=request_errors,
    started_at=started_at.isoformat(),
    finished_at=finished_at.isoformat(),
)

first_summary = {
    "source": "choice",
    "requested_symbols": SYMBOLS,
    "requested_report_dates": REPORT_DATES,
    "selected_indicators": selected_indicators,
    "rejected_indicators": rejected_indicators,
    "statement_received_rows": len(statement_records),
    "statement_stored_rows": first_statement_stored,
    "dividend_received_rows": len(dividend_records),
    "dividend_stored_rows": first_dividend_stored,
    "errors": request_errors,
    "started_at": started_at.isoformat(),
    "finished_at": finished_at.isoformat(),
}

print(json.dumps(first_summary, ensure_ascii=False, indent=2, default=str))
display(request_audit)


[EmQuantAPI Python] [Em_Info][2026-09-02 13:24:42]:The current version is EmQuantAPI(V2.7.5.0).

[EmQuantAPI Python] [Em_Info][2026-09-02 13:24:42]:verifying your token...

[EmQuantAPI Python] [Em_Info][2026-09-02 13:24:42]:connect server...

[EmQuantAPI Python] [Em_Info][2026-09-02 13:24:46]:token login start success!

[EmQuantAPI Python] [Em_Info][2026-09-02 13:24:50]:loading ChoiceToHQ.xml...

[EmQuantAPI Python] [Em_Info][2026-09-02 13:25:25]:heartbeatthread end.

{
  "source": "choice",
  "requested_symbols": [
    "000001.SZ",
    "600519.SH",
    "300750.SZ"
  ],
  "requested_report_dates": [
    "2025-12-31",
    "2026-06-30"
  ],
  "selected_indicators": {
    "income": [
      "NETPROFIT",
      "OPERATEREVE",
      "PARENTNETPROFIT"
    ],
    "balance": [
      "SUMASSET",
      "SUMLIAB",
      "SUMSHEQUITY"
    ],
    "cashflow": [
      "NETFINACASHFLOW",
      "NETINVCASHFLOW",
      "NETOPERATECASHFLOW",
      "NICASHEQUI"
    ],
    "dividend": [
      "DIVCAPITPSRATI

,request,ctr_name,status,rows,error_code,error,options
0,income:000001.SZ:2025-12-31,IncomeStatementSHSZ,success,1,0,,"SecuCode=000001.SZ,ReportDate=2025-12-31,Repor..."
1,balance:000001.SZ:2025-12-31,BalanceStatementSHSZ,success,1,0,,"SecuCode=000001.SZ,ReportDate=2025-12-31,Repor..."
2,cashflow:000001.SZ:2025-12-31,CashFlowStatementSHSZ,success,1,0,,"SecuCode=000001.SZ,ReportDate=2025-12-31,Repor..."
3,income:000001.SZ:2026-06-30,IncomeStatementSHSZ,success,1,0,,"SecuCode=000001.SZ,ReportDate=2026-06-30,Repor..."
4,balance:000001.SZ:2026-06-30,BalanceStatementSHSZ,success,1,0,,"SecuCode=000001.SZ,ReportDate=2026-06-30,Repor..."
5,cashflow:000001.SZ:2026-06-30,CashFlowStatementSHSZ,success,1,0,,"SecuCode=000001.SZ,ReportDate=2026-06-30,Repor..."
6,dividend:000001.SZ:2025-12-31:2026-06-30,DividendImplementationInfo,success,1,0,,"secucode=000001.SZ,StartDate=2025-12-31,EndDat..."
7,income:600519.SH:2025-12-31,IncomeStatementSHSZ,success,1,0,,"SecuCode=600519.SH,ReportDate=2025-12-31,Repor..."
8,balance:600519.SH:2025-12-31,BalanceStatementSHSZ,success,1,0,,"SecuCode=600519.SH,ReportDate=2025-12-31,Repor..."
9,cashflow:600519.SH:2025-12-31,CashFlowStatementSHSZ,success,1,0,,"SecuCode=600519.SH,ReportDate=2025-12-31,Repor..."


## 读取SQLite证据


In [18]:
database = Database(DATABASE_PATH)
statement_facts = database.query_financial_statement_facts(
    source="choice", symbols=SYMBOLS,
    start_report_date=min(REPORT_DATES), end_report_date=max(REPORT_DATES),
)
statement_facts = statement_facts[statement_facts["report_date"].isin(REPORT_DATES)].copy()
allowed_statement_fields = {
    (statement_type, indicator)
    for statement_type, spec in REPORT_SPECS.items()
    for indicator in spec["indicators"]
}
statement_facts = statement_facts[
    statement_facts.apply(
        lambda row: (row["statement_type"], row["indicator"]) in allowed_statement_fields,
        axis=1,
    )
].copy()

dividend_facts = database.query_dividend_facts(
    source="choice", symbols=SYMBOLS,
    start_report_date=min(REPORT_DATES), end_report_date=max(REPORT_DATES),
)
dividend_facts = dividend_facts[dividend_facts["report_date"].isin(REPORT_DATES)].copy()
dividend_facts = dividend_facts[
    dividend_facts["indicator"].isin(DIVIDEND_SPEC["indicators"])
].copy()

ingestion_runs = database.query_financial_ingestion_runs().tail(10).copy()
idempotency = pd.DataFrame([counts_before, counts_after_first, counts_after_second])

selected_rows = [
    {"dataset": dataset, "indicator": indicator, "status": "selected"}
    for dataset, indicators in selected_indicators.items() for indicator in indicators
]
selected_frame = pd.DataFrame(selected_rows, columns=["dataset", "indicator", "status"])

rejected_rows = [
    {"dataset": dataset, "indicator": indicator, "status": "missing", "reason": reason}
    for dataset, fields in rejected_indicators.items() for indicator, reason in fields.items()
]
rejected_frame = pd.DataFrame(rejected_rows, columns=["dataset", "indicator", "status", "reason"])
errors_frame = pd.DataFrame(
    [{"request": key, "error": value} for key, value in request_errors.items()],
    columns=["request", "error"],
)

print("财务事实：", len(statement_facts))
print("分红事实：", len(dividend_facts))
display(idempotency)
display(selected_frame)
display(rejected_frame)
display(errors_frame)


财务事实： 60
分红事实： 44


,stage,statement_scope,dividend_scope,financial_run_log
0,刷新前,60,44,3
1,第一次落库后,60,44,3
2,第二次本地upsert后,60,44,3


,dataset,indicator,status
0,income,NETPROFIT,selected
1,income,OPERATEREVE,selected
2,income,PARENTNETPROFIT,selected
3,balance,SUMASSET,selected
4,balance,SUMLIAB,selected
5,balance,SUMSHEQUITY,selected
6,cashflow,NETFINACASHFLOW,selected
7,cashflow,NETINVCASHFLOW,selected
8,cashflow,NETOPERATECASHFLOW,selected
9,cashflow,NICASHEQUI,selected


,dataset,indicator,status,reason


,request,error


## 完整性与空值分析

财务报表要求覆盖全部样本和报告期。分红属于事件数据，不要求每只证券、每个报告期都有记录；“查询成功但无事件”与接口失败分开记录。


In [19]:
statement_facts["has_value"] = statement_facts["value_numeric"].notna() | statement_facts["value_text"].notna()
dividend_facts["has_value"] = dividend_facts["value_numeric"].notna() | dividend_facts["value_text"].notna()

statement_completeness = (
    statement_facts.groupby(["statement_type", "indicator"], dropna=False)
    .agg(rows=("symbol", "size"), symbols=("symbol", "nunique"), report_dates=("report_date", "nunique"), non_null=("has_value", "sum"))
    .reset_index()
) if not statement_facts.empty else pd.DataFrame(columns=["statement_type", "indicator", "rows", "symbols", "report_dates", "non_null"])

dividend_completeness = (
    dividend_facts.groupby("indicator", dropna=False)
    .agg(rows=("symbol", "size"), symbols=("symbol", "nunique"), report_dates=("report_date", "nunique"), non_null=("has_value", "sum"))
    .reset_index()
) if not dividend_facts.empty else pd.DataFrame(columns=["indicator", "rows", "symbols", "report_dates", "non_null"])

display(statement_completeness)
display(dividend_completeness)


,statement_type,indicator,rows,symbols,report_dates,non_null
0,balance,SUMASSET,6,3,2,6
1,balance,SUMLIAB,6,3,2,6
2,balance,SUMSHEQUITY,6,3,2,6
3,cashflow,NETFINACASHFLOW,6,3,2,6
4,cashflow,NETINVCASHFLOW,6,3,2,6
5,cashflow,NETOPERATECASHFLOW,6,3,2,6
6,cashflow,NICASHEQUI,6,3,2,6
7,income,NETPROFIT,6,3,2,6
8,income,OPERATEREVE,6,3,2,6
9,income,PARENTNETPROFIT,6,3,2,6


,indicator,rows,symbols,report_dates,non_null
0,DIVCAPITPSRATIO,4,3,2,0
1,DIVCASHPSAFTAX,4,3,2,4
2,DIVCASHPSBFTAX,4,3,2,4
3,DIVEXDATE,4,3,2,4
4,DIVIMPLANNCDATE,4,3,2,4
5,DIVPAYDATE,4,3,2,4
6,DIVRECORDDATE,4,3,2,4
7,DIVRTISSBASESHARES,4,3,2,4
8,DIVSTOCKPSRATIO,4,3,2,0
9,DIVWAY,4,3,2,4


## 质量门槛


In [20]:
expected_statement_rows = len(SYMBOLS) * len(REPORT_DATES) * sum(
    len(spec["indicators"]) for spec in REPORT_SPECS.values()
)
expected_financial_requests = len(SYMBOLS) * len(REPORT_DATES) * len(REPORT_SPECS)
expected_dividend_requests = len(SYMBOLS)

statement_duplicates = int(statement_facts.duplicated(
    ["source", "symbol", "statement_type", "report_date", "indicator"]
).sum()) if not statement_facts.empty else 0
dividend_duplicates = int(dividend_facts.duplicated(
    ["source", "symbol", "report_date", "indicator"]
).sum()) if not dividend_facts.empty else 0

numeric_values = pd.concat(
    [statement_facts.get("value_numeric", pd.Series(dtype=float)), dividend_facts.get("value_numeric", pd.Series(dtype=float))],
    ignore_index=True,
).dropna()
finite_numeric = all(math.isfinite(float(value)) for value in numeric_values)
core_unchanged = RUN_IDEMPOTENCY_CHECK and all(
    counts_after_first[key] == counts_after_second[key]
    for key in ["statement_scope", "dividend_scope"]
)
financial_audit = request_audit[request_audit["request"].str.startswith(("income:", "balance:", "cashflow:"))]
dividend_audit = request_audit[request_audit["request"].str.startswith("dividend:")]

gates = []
def add_gate(name, passed, evidence):
    gates.append({"质量门槛": name, "通过": bool(passed), "证据": str(evidence)})

add_gate("数据库路径为项目主库", DATABASE_PATH.parent == PROJECT_ROOT / "data", str(DATABASE_PATH))
add_gate("财务CTR请求数完整", len(financial_audit) == expected_financial_requests, f"actual={len(financial_audit)}, expected={expected_financial_requests}")
add_gate("分红CTR请求数完整", len(dividend_audit) == expected_dividend_requests, f"actual={len(dividend_audit)}, expected={expected_dividend_requests}")
add_gate("所有CTR请求未返回错误", not (request_audit["status"] == "error").any() and not request_errors, request_errors or "errors={}")
add_gate("三类财务字段全部返回", all(len(selected_indicators[key]) == len(REPORT_SPECS[key]["indicators"]) for key in REPORT_SPECS), selected_indicators)
add_gate("财务事实行数符合预期", len(statement_facts) == expected_statement_rows and expected_statement_rows > 0, f"actual={len(statement_facts)}, expected={expected_statement_rows}")
add_gate("财务覆盖全部样本证券", statement_facts["symbol"].nunique() == len(SYMBOLS) if not statement_facts.empty else False, f"actual={statement_facts['symbol'].nunique() if not statement_facts.empty else 0}, expected={len(SYMBOLS)}")
add_gate("财务覆盖全部报告期", statement_facts["report_date"].nunique() == len(REPORT_DATES) if not statement_facts.empty else False, f"actual={statement_facts['report_date'].nunique() if not statement_facts.empty else 0}, expected={len(REPORT_DATES)}")
add_gate("财务覆盖三张报表", set(statement_facts["statement_type"]) == {"income", "balance", "cashflow"} if not statement_facts.empty else False, sorted(statement_facts["statement_type"].unique()) if not statement_facts.empty else [])
add_gate("每类财务报表均存在非空值", all(
    int(statement_facts.loc[statement_facts["statement_type"] == key, "has_value"].sum()) > 0
    for key in REPORT_SPECS
), statement_completeness.to_dict("records"))
add_gate("分红专题报表调用成功", len(dividend_audit) == expected_dividend_requests and not (dividend_audit["status"] == "error").any(), dividend_audit[["request", "status", "rows"]].to_dict("records"))
add_gate("样本范围存在分红事件", not dividend_facts.empty and int(dividend_facts["has_value"].sum()) > 0, f"rows={len(dividend_facts)}, non_null={int(dividend_facts['has_value'].sum()) if not dividend_facts.empty else 0}")
add_gate("财务事实无重复主键", statement_duplicates == 0, f"duplicates={statement_duplicates}")
add_gate("分红事实无重复主键", dividend_duplicates == 0, f"duplicates={dividend_duplicates}")
add_gate("来源均为Choice", set(statement_facts.get("source", [])) <= {"choice"} and set(dividend_facts.get("source", [])) <= {"choice"}, "source=choice")
add_gate("财务单位为官方元口径", set(statement_facts.get("unit", [])) <= {"CNY"}, sorted(set(statement_facts.get("unit", []))))
add_gate("数值字段无NaN或无穷值", finite_numeric, f"numeric_checked={len(numeric_values)}")
add_gate("第二次本地upsert核心表不增长", core_unchanged, idempotency.to_dict("records"))
add_gate("SQLite完整性检查通过", scalar("PRAGMA quick_check") == "ok", scalar("PRAGMA quick_check"))

quality_gates = pd.DataFrame(gates)
passed_count = int(quality_gates["通过"].sum())
failed_count = int((~quality_gates["通过"]).sum())
print(f"质量门槛：{passed_count}项通过，{failed_count}项失败")
display(quality_gates)


质量门槛：19项通过，0项失败


,质量门槛,通过,证据
0,数据库路径为项目主库,True,D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\data\...
1,财务CTR请求数完整,True,"actual=18, expected=18"
2,分红CTR请求数完整,True,"actual=3, expected=3"
3,所有CTR请求未返回错误,True,errors={}
4,三类财务字段全部返回,True,"{'income': ['NETPROFIT', 'OPERATEREVE', 'PAREN..."
5,财务事实行数符合预期,True,"actual=60, expected=60"
6,财务覆盖全部样本证券,True,"actual=3, expected=3"
7,财务覆盖全部报告期,True,"actual=2, expected=2"
8,财务覆盖三张报表,True,"['balance', 'cashflow', 'income']"
9,每类财务报表均存在非空值,True,"[{'statement_type': 'balance', 'indicator': 'S..."


## 导出Excel和JSON证据


In [21]:
data_map = pd.DataFrame([
    {"dataset": "financial_statement_fact", "grain": "source+symbol+statement_type+report_date+indicator", "purpose": "Choice CTR三张财务报表核心事实", "unit_policy": "官方字段标注为元，unit=CNY", "source": "choice"},
    {"dataset": "dividend_fact", "grain": "source+symbol+report_date+indicator", "purpose": "Choice DividendImplementationInfo分红实施事实", "unit_policy": "按字段保存CNY/share、10k_share、date、text或vendor_raw_ratio", "source": "choice"},
    {"dataset": "financial_ingestion_run", "grain": "run_id", "purpose": "CTR下载、字段返回和错误审计", "unit_policy": "不适用", "source": "choice"},
])

official_mapping = pd.DataFrame([
    {"dataset": key, "function": "ctr", "ctr_name": spec["ctr_name"], "parameters": "SecuCode,ReportDate,ReportType", "indicators": ",".join(spec["indicators"])}
    for key, spec in REPORT_SPECS.items()
] + [{
    "dataset": "dividend", "function": "ctr", "ctr_name": DIVIDEND_SPEC["ctr_name"],
    "parameters": "secucode,StartDate,EndDate,DateType=3",
    "indicators": ",".join(DIVIDEND_SPEC["indicators"]),
}])

overview = pd.DataFrame([
    {"项目": "运行时间", "值": datetime.now(timezone.utc).isoformat()},
    {"项目": "Notebook版本", "值": "0.7.1-ctr"},
    {"项目": "数据库", "值": str(DATABASE_PATH)},
    {"项目": "备份", "值": str(backup_path) if backup_path else "未创建"},
    {"项目": "证券", "值": ",".join(SYMBOLS)},
    {"项目": "报告期", "值": ",".join(REPORT_DATES)},
    {"项目": "Choice请求次数", "值": len(request_audit)},
    {"项目": "财务事实", "值": len(statement_facts)},
    {"项目": "分红事实", "值": len(dividend_facts)},
    {"项目": "质量门槛", "值": f"{passed_count}通过/{failed_count}失败"},
])

excel_path = OUTPUT_DIR / f"Choice财务报表分红小样本验收_{run_timestamp}.xlsx"
json_path = OUTPUT_DIR / f"Choice财务报表分红小样本验收_{run_timestamp}.json"

sheets = {
    "验收总览": overview,
    "质量门槛": quality_gates,
    "落库计数": idempotency,
    "官方接口映射": official_mapping,
    "请求审计": request_audit,
    "选中指标": selected_frame,
    "缺失字段": rejected_frame,
    "请求错误": errors_frame,
    "财务事实": statement_facts.drop(columns=["has_value"], errors="ignore"),
    "分红事实": dividend_facts.drop(columns=["has_value"], errors="ignore"),
    "财务完整性": statement_completeness,
    "分红完整性": dividend_completeness,
    "运行日志": ingestion_runs,
    "数据地图": data_map,
}

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for sheet_name, frame in sheets.items():
        frame.to_excel(writer, sheet_name=sheet_name[:31], index=False)

from openpyxl import load_workbook
workbook = load_workbook(excel_path)
for worksheet in workbook.worksheets:
    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = worksheet.dimensions
    for column_cells in worksheet.columns:
        values = [str(cell.value or "") for cell in list(column_cells)[:200]]
        width = min(max(max((len(value) for value in values), default=8) + 2, 10), 60)
        worksheet.column_dimensions[column_cells[0].column_letter].width = width
workbook.save(excel_path)

json_payload = {
    "metadata": safe_config,
    "ingestion": first_summary,
    "quality_summary": {"passed": passed_count, "failed": failed_count},
    "quality_gates": quality_gates.to_dict("records"),
    "idempotency_counts": idempotency.to_dict("records"),
    "official_mapping": official_mapping.to_dict("records"),
    "request_audit": request_audit.to_dict("records"),
    "selected_indicators": selected_frame.to_dict("records"),
    "missing_fields": rejected_frame.to_dict("records"),
    "errors": errors_frame.to_dict("records"),
    "statement_completeness": statement_completeness.to_dict("records"),
    "dividend_completeness": dividend_completeness.to_dict("records"),
    "data_map": data_map.to_dict("records"),
}
json_path.write_text(json.dumps(json_payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

print("Excel：", excel_path)
print("JSON：", json_path)


Excel： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice财务报表分红小样本验收_20260902_132440.xlsx
JSON： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice财务报表分红小样本验收_20260902_132440.json


## 结果解释

- 财务请求失败：先查看“请求审计”的`ErrorCode/ErrorMsg`，不要反复重试登录。
- 分红请求为`success_empty`：接口调用成功，但指定报告期没有匹配的已实施分红事件，不等同于权限失败。
- “样本范围存在分红事件”未通过：先在Choice终端核对3只样本对应报告期是否已经实施分红；无需把空值填0。
- 少数字段缺失：查看“缺失字段”，可能是账号产品权限或报表版本差异；已成功返回的字段仍会落库。
- 全部通过：说明官方CTR调用、三张财务报表、分红实施、SQLite落库和幂等性均已在小样本范围成立。


In [22]:
print(f"最终结论：{passed_count}项通过，{failed_count}项失败")
if failed_count == 0:
    print("✅ 09号Choice财务报表与分红小样本验收通过。")
else:
    print("⚠️ 已保存验收证据，请查看‘质量门槛’、‘请求审计’和‘请求错误’。")

if STRICT_MODE and failed_count:
    failed_names = quality_gates.loc[~quality_gates["通过"], "质量门槛"].tolist()
    raise RuntimeError(f"严格模式：以下质量门槛未通过：{failed_names}")


最终结论：19项通过，0项失败
✅ 09号Choice财务报表与分红小样本验收通过。
